# GAN — Generative Adversarial Nets (2014)

_Papers · Generative Models_

**Train a generator by pitting it against a discriminator that tries to tell real data from its fakes.**

---

This notebook is a practice scaffold for the **GAN — Generative Adversarial Nets (2014)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns**. We pull a few raw samples from the MNIST dataset to see them before any transforms.

In [ ]:
import torchvision

preview = torchvision.datasets.MNIST(root="./data", train=True, download=True)
print("dataset: MNIST   samples:", len(preview))
first_image, first_label = preview[0]
print("one sample:", first_image.size, "image,  label =", first_label)
print("classes:", getattr(preview, "classes", "(digit labels 0-9)"))
samples = [preview[i] for i in range(5)]
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn as nn, math
import torchvision, torchvision.transforms as T

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

# --- 0. Sanity-check the worked example (Proposition 1 + the global optimum). ---
pdata, pg = 0.6, 0.2
Dstar = pdata / (pdata + pg)                 # D*_G(x) = p_data/(p_data+p_g)
print("worked example:")
print("  D*(x) = %.2f/%.2f = %.4f" % (pdata, pdata+pg, Dstar))           # 0.75
print("  log D*(x)        = %.4f" % math.log(Dstar))                     # -0.2877
print("  log(1-0.3)       = %.4f" % math.log(1 - 0.3))                   # -0.3567
print("  global optimum -log4 = %.4f" % (-math.log(4)))                  # -1.3863
print("  D-loss at equilibrium 2*log2 = %.4f" % (2*math.log(2)))        # 1.3863


# --- 1. Generator and discriminator, built by hand from nn.Linear. ---
Z = 64
class G(nn.Module):                          # noise z -> 784 pixels in [-1,1]
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(Z, 256), nn.LeakyReLU(0.2),
            nn.Linear(256, 512), nn.LeakyReLU(0.2),
            nn.Linear(512, 784), nn.Tanh())
    def forward(self, z): return self.net(z)

class D(nn.Module):                          # 784 pixels -> 1 real/fake logit
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 512), nn.LeakyReLU(0.2),
            nn.Linear(512, 256), nn.LeakyReLU(0.2),
            nn.Linear(256, 1))               # logit; sigmoid lives inside the BCE loss
    def forward(self, x): return self.net(x)

gen, dis = G().to(device), D().to(device)
bce = nn.BCEWithLogitsLoss()
optG = torch.optim.Adam(gen.parameters(), lr=2e-4, betas=(0.5, 0.999))
optD = torch.optim.Adam(dis.parameters(), lr=2e-4, betas=(0.5, 0.999))


# --- 2. MNIST (torchvision is preinstalled in Colab). Scale to [-1,1] to match tanh. ---
tfm = T.Compose([T.ToTensor(), T.Normalize((0.5,), (0.5,))])
data = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=tfm)
loader = torch.utils.data.DataLoader(data, batch_size=128, shuffle=True, drop_last=True)


# --- 3. Print a tiny ASCII preview of generated digits so we SEE them improve. ---
fixed_z = torch.randn(1, Z, device=device)
def preview(tag):
    gen.eval()
    with torch.no_grad():
        img = gen(fixed_z).view(28, 28).cpu()
    gen.train()
    img = (img + 1) / 2                       # back to [0,1]
    rows = []
    for r in range(0, 28, 2):                 # downsample to 14 rows for the console
        line = "".join(" .:-=+*#%@"[min(9, int(img[r, c].clamp(0,1) * 9))] for c in range(0, 28, 1))
        rows.append(line)
    print("--- samples @", tag, "---")
    print("\n".join(rows))


# --- 4. The adversarial training loop (the novel part). ---
def train(epochs=3):
    step = 0; dl_hist, gl_hist = [], []
    for ep in range(epochs):
        for x, _ in loader:
            x = x.view(x.size(0), -1).to(device)
            m = x.size(0)
            real_t = torch.ones(m, 1, device=device)
            fake_t = torch.zeros(m, 1, device=device)

            # (a) DISCRIMINATOR step (k=1): ascend log D(x) + log(1-D(G(z))).
            z = torch.randn(m, Z, device=device)
            fake = gen(z).detach()            # detach: this step must NOT move G
            lossD = bce(dis(x), real_t) + bce(dis(fake), fake_t)
            optD.zero_grad(); lossD.backward(); optD.step()

            # (b) GENERATOR step: non-saturating -- maximize log D(G(z)).
            z = torch.randn(m, Z, device=device)
            lossG = bce(dis(gen(z)), real_t)  # wants D to call fakes "real"
            optG.zero_grad(); lossG.backward(); optG.step()

            if step % 200 == 0:
                dl_hist.append((step, lossD.item())); gl_hist.append((step, lossG.item()))
                if step % 600 == 0:
                    print("step %5d   D loss %.3f   G loss %.3f" % (step, lossD.item(), lossG.item()))
            step += 1
        preview("epoch %d" % ep)              # watch the digit sharpen each epoch
    return dl_hist, gl_hist

print("\nbefore training:"); preview("init")
dl_hist, gl_hist = train(epochs=3)
print("\nD loss should settle near 2*log2 = 1.386 (D reduced to guessing); samples sharpen each epoch.")
print("If every sample looks like the SAME digit -> mode collapse (the classic GAN pitfall).")
# (Exact numbers vary by hardware/seed; this is our small run, not the paper's reported result.)

## Visualize the data & results

_As a GAN trains, where do the discriminator and generator losses go — and does that match the theory (D → guessing, value → -log 4)?_

In [ ]:
import torch, torch.nn as nn, numpy as np
torch.manual_seed(0)

# Tiny GAN on a 2-D toy blob. Same loop as the MNIST notebook (G/D from nn.Linear,
# non-saturating G loss), small enough to run in seconds and show the equilibrium.
def real_batch(m):                            # real data: a Gaussian blob
    return torch.randn(m, 2) * 0.4 + torch.tensor([1.5, 1.0])

Z = 8
G = nn.Sequential(nn.Linear(Z, 32), nn.ReLU(), nn.Linear(32, 2))
D = nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 1))
bce  = nn.BCEWithLogitsLoss()
optD = torch.optim.Adam(D.parameters(), lr=2e-3, betas=(0.5, 0.999))
optG = torch.optim.Adam(G.parameters(), lr=2e-3, betas=(0.5, 0.999))

m, dl, gl = 128, [], []
for step in range(600):
    x = real_batch(m); z = torch.randn(m, Z)
    fake = G(z).detach()                                   # detach in the D step
    lossD = bce(D(x), torch.ones(m,1)) + bce(D(fake), torch.zeros(m,1))
    optD.zero_grad(); lossD.backward(); optD.step()

    z = torch.randn(m, Z)
    lossG = bce(D(G(z)), torch.ones(m,1))                  # non-saturating: max log D(G(z))
    optG.zero_grad(); lossG.backward(); optG.step()
    dl.append(lossD.item()); gl.append(lossG.item())

idx = np.linspace(0, 599, 30).astype(int)
print("Discriminator:", [[int(i), round(dl[i], 4)] for i in idx])
print("Generator    :", [[int(i), round(gl[i], 4)] for i in idx])
with torch.no_grad():
    print("real mean [1.5, 1.0]  vs  gen mean:",
          [round(v, 3) for v in G(torch.randn(2000, Z)).mean(0).tolist()])
# D loss -> ~1.386 = 2*log2 = -log4 (D reduced to guessing); gen mean -> real mean.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The detach ablation. In your working GAN, remove the .detach() from the fakes in
            the discriminator step (so D's loss now flows into G too). Keep
            everything else identical. What goes wrong, and what does that show about which step is allowed to
            move $G$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Change the D-step from bce(D(G(z).detach()), 0) to bce(D(G(z)), 0); leave the G-step alone. — _An honest ablation changes exactly one thing &mdash; whether the discriminator's gradient reaches the generator._
- Retrain and watch the samples: $G$ now receives a "make the fake look more fake" signal during the D-step, fighting its own G-step. — _The D-step loss rewards $D$ for calling fakes "fake"; backprop into $G$ pushes $G$ to comply &mdash; the opposite of what $G$ should learn._
- Conclude that only the generator step may update $G$; the discriminator step must treat the fakes as fixed inputs. — _Detaching severs the graph at $G(z)$ so the D-step updates only $\theta_d$, keeping the two objectives clean._

**Answer:** Without .detach(), the discriminator step also back-propagates into $G$ with a
                 signal that wants the fake judged "fake" &mdash; directly undoing the generator step. Training
                 destabilizes or stalls and samples fail to improve. The fix isolates responsibilities: the
                 D-step updates only $D$ (fakes detached), the G-step updates only $G$. This shows the two
                 alternating optimizations must not share gradients through $G(z)$.

</details>

**Problem 2.** Your worked example had $p_{\text{data}}(x) = 0.6$ and $p_g(x) = 0.2$, giving
            $D^*_G(x) = 0.75$. Suppose instead the generator has matched the data at this point, so
            $p_g(x) = 0.6$. What is $D^*_G(x)$ now, and what does that say about the discriminator at
            convergence?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Plug into Proposition 1: $D^*_G(x) = \dfrac{0.6}{0.6 + 0.6} = \dfrac{0.6}{1.2} = 0.5$. — _When the real and fake densities are equal at $x$, the optimal detector has no information to tell them apart._
- Note that if $p_g = p_{\text{data}}$ everywhere, then $D^* = 0.5$ for every $x$. — _The discriminator is reduced to a coin flip &mdash; exactly the equilibrium of the game._
- Map that to the loss: each BCE term becomes $-\log 0.5 = \log 2 \approx 0.693$, so $D$'s total loss settles near $2\log 2 \approx 1.386 = -\log 4$ in value-function terms. — _This is why a converged GAN does NOT drive $D$'s loss to zero._

**Answer:** $D^*_G(x) = 0.6/(0.6+0.6) = 0.5$. Once the generator matches the data ($p_g = p_{\text{data}}$),
                 the best possible discriminator outputs $0.5$ everywhere &mdash; it can only guess. Its loss
                 settles near $2\log 2 \approx 1.386$ (the optimum $-\log 4$), the flat plateau you see in the
                 loss panel, not $0$.

</details>

**Problem 3.** You print samples during training and notice that after a while every generated image is the
            same digit (say, a "1"), even though the loss curves look stable. What failure is this, why does it
            still fool the discriminator, and what is one mitigation?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Name it: this is mode collapse &mdash; $G$ covers only one mode of $p_{\text{data}}$ instead of all ten digits. — _The data has many modes (digit classes); $G$ has parked $p_g$ on a single one._
- Explain why it persists: a single very-realistic "1" can make $D(G(z))$ high, so the generator's loss is low even though variety is gone. — _The minimax objective rewards fooling $D$ per-sample; it does not directly force $G$ to cover every mode._
- Apply a mitigation: lower/asymmetric learning rates, minibatch features, or a different loss (e.g. Wasserstein GAN). For this lesson, just flag it on the sample grid. — _Later papers in this module add explicit fixes; recognizing collapse is the first skill._

**Answer:** This is mode collapse: the generator covers only one mode of the data while ignoring
                 the rest. It persists because fooling $D$ on each individual sample &mdash; not covering every
                 mode &mdash; is what the loss rewards, so a single convincing digit keeps $G$'s loss low.
                 Mitigations include smaller/asymmetric learning rates, minibatch discrimination, or switching to
                 a Wasserstein loss; the classic GAN loop has no built-in guard against it.

</details>